# 04 — Decision Tree

Train a decision tree using the saved training–validation split and the same 41 input features as logistic regression.

One-hot encode categorical features and keep numerical features in their original units. Compare training and validation performance, then inspect false alarms and missed attacks.

The official test set remains reserved for final evaluation.

## 1. Recover the saved split

Load the official training file and recover the records assigned to each partition. Reusing the saved IDs keeps this experiment comparable with logistic regression.

The assertions check row counts and ensure that no record ID belongs to both partitions.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent

# Load the original development data and saved split assignments.
official_train_df = pd.read_csv(project_root / "data" / "raw" / "UNSW_NB15_training-set.csv")
split_assignments = pd.read_csv(project_root / "data" / "processed" / "train_validation_split.csv")

# Get the record IDs assigned to each partition.
fit_ids = split_assignments.loc[
    split_assignments["partition"].eq("training"), "id"
]
validation_ids = split_assignments.loc[
    split_assignments["partition"].eq("validation"), "id"
]

# Recover the corresponding records.
fit_df = official_train_df.loc[
    official_train_df["id"].isin(fit_ids)
].copy()

validation_df = official_train_df.loc[
    official_train_df["id"].isin(validation_ids)
].copy()

# Check that all assigned records were recovered without overlapping IDs.
assert len(fit_df) == len(fit_ids)
assert len(validation_df) == len(validation_ids)
assert set(fit_ids).isdisjoint(validation_ids)
assert len(fit_df) + len(validation_df) == len(official_train_df)

## 2. Define inputs and target

`X` contains traffic measurements; `y` contains the binary answer (`0 = normal`, `1 = attack`).

Exclude `id` because it is a record identifier, and exclude `label` and `attack_cat` because they contain the answers. Also exclude `is_ftp_login`: it duplicates `ct_ftp_cmd` in the development data and has an unresolved discrepancy with its documented binary meaning.

Expected input shapes: **(140272, 41)** for training and **(35069, 41)** for validation. The 41 inputs contain 38 numerical and 3 categorical columns.

In [2]:
# Confirm the duplicate-feature finding in the training partition.
assert fit_df["is_ftp_login"].eq(fit_df["ct_ftp_cmd"]).all()

excluded_columns = ["id", "attack_cat", "label", "is_ftp_login"]

# X contains the traffic measurements supplied to the model.
X_fit = fit_df.drop(columns=excluded_columns)
X_validation = validation_df.drop(columns=excluded_columns)

# y contains the corresponding correct answers.
y_fit = fit_df["label"].copy()
y_validation = validation_df["label"].copy()

print("X_fit:", X_fit.shape)
print("y_fit:", y_fit.shape)
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

X_fit: (140272, 41)
y_fit: (140272,)
X_validation: (35069, 41)
y_validation: (35069,)


In [3]:
# These text columns need categorical encoding.
categorical_columns = ["proto", "service", "state"]

# Select the numerical measurements by their stored data types.
numerical_columns = X_fit.select_dtypes(
    include="number"
).columns.tolist()

# Ensure that every input column has been accounted for.
assert set(categorical_columns + numerical_columns) == set(X_fit.columns)

print("Categorical columns:", categorical_columns)
print("Numerical column count:", len(numerical_columns))

Categorical columns: ['proto', 'service', 'state']
Numerical column count: 38


## 3. Import tools and define evaluation

A pipeline keeps preprocessing and classification together. The evaluation function uses the confusion matrix to calculate accuracy, attack precision, attack recall, and the false-positive rate.

Its results are proportions between 0 and 1. We multiply by 100 only when displaying percentages.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

In [5]:
def summarize_predictions(y_true, y_pred):
    # With labels [0, 1], the four cells are TN, FP, FN, TP.
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return {
        "accuracy": (tp + tn) / (tn + fp + fn + tp),
        "precision": tp / (tp + fp) if tp + fp else 0.0,
        "recall": tp / (tp + fn) if tp + fn else 0.0,
        "false_positive_rate": fp / (tn + fp) if tn + fp else 0.0,
    }

## 4. Train a decision tree

A tree learns a sequence of feature-based questions. It chooses splits that separate the training labels into less mixed groups. Each final group is a **leaf**, which predicts its majority class.

Trees do not need standard scaling. `passthrough` keeps numerical measurements in their original units; one-hot encoding converts the text categories into indicator columns. `handle_unknown="ignore"` lets the encoder process an unseen category using all zeros for that feature's indicator columns.

Our initial settings are:

- `max_depth=8`: at most eight splits along a path.
- `min_samples_leaf=20`: at least 20 training records per leaf.
- `random_state=42`: make randomized choices reproducible.

These are starting settings, not optimized values. Limiting tree complexity helps reduce the risk of overfitting.

In [6]:
# Trees can use numerical measurements in their original units.
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", numerical_columns),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns,
        ),
    ]
)

# Limit tree complexity to reduce the risk of overfitting.
tree_model = Pipeline(
    steps=[
        ("preprocessing", tree_preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                max_depth=8,
                min_samples_leaf=20,
                random_state=42,
            ),
        ),
    ]
)

# Learn preprocessing and tree rules using training records only.
tree_model.fit(X_fit, y_fit)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](41,)","['dur','proto','service',...,'ct_src_ltm','ct_srv_dst','is_sm_ips_ports']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,41
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='

## 5. Evaluate training and validation performance

Use the same fitted tree to predict both partitions. `predict()` applies the learned preprocessing and rules without retraining.

Compare the two rows for a training–validation gap. In the confusion matrix, rows are actual labels and columns are predicted labels. A normal record predicted as an attack is a **false positive**; an attack predicted as normal is a **false negative**.

In [7]:
# Evaluate the same fitted tree on both partitions.
tree_training_predictions = tree_model.predict(X_fit)
tree_predictions = tree_model.predict(X_validation)

tree_partition_results = pd.DataFrame(
    {
        "Training": summarize_predictions(
            y_fit, tree_training_predictions
        ),
        "Validation": summarize_predictions(
            y_validation, tree_predictions
        ),
    }
).T

# Show metric values as percentages.
display((tree_partition_results * 100).round(2))

# Count correct predictions, false alarms, and missed attacks.
tree_confusion = pd.DataFrame(
    confusion_matrix(y_validation, tree_predictions, labels=[0, 1]),
    index=["Actual normal", "Actual attack"],
    columns=["Predicted normal", "Predicted attack"],
)

display(tree_confusion)

,accuracy,precision,recall,false_positive_rate
Training,94.18,94.68,96.89,11.60
Validation,94.39,94.77,97.11,11.42


,Predicted normal,Predicted attack
Actual normal,9921,1279
Actual attack,689,23180


In [8]:
from sklearn.metrics import roc_auc_score, average_precision_score

# Find the probability column for attack (label 1).
tree_attack_column = list(tree_model.classes_).index(1)

# Get validation probability scores from the fitted pipeline.
tree_attack_probabilities = tree_model.predict_proba(
    X_validation
)[:, tree_attack_column]

# Calculate F1 from the default predictions.
tn, fp, fn, tp = confusion_matrix(
    y_validation, tree_predictions, labels=[0, 1]
).ravel()

tree_f1 = 2 * tp / (2 * tp + fp + fn)

# Evaluate ranking across thresholds using probability scores.
tree_roc_auc = roc_auc_score(
    y_validation, tree_attack_probabilities
)
tree_ap = average_precision_score(
    y_validation, tree_attack_probabilities
)

print(f"Decision-tree validation F1:      {tree_f1:.2%}")
print(f"Decision-tree validation ROC AUC: {tree_roc_auc:.4f}")
print(f"Decision-tree validation AP:      {tree_ap:.4f}")

Decision-tree validation F1:      95.93%
Decision-tree validation ROC AUC: 0.9872
Decision-tree validation AP:      0.9928


### Decision-tree findings

The initial tree uses max_depth=8 and min_samples_leaf=20.

- Training accuracy: 94.18%.
- Validation accuracy: 94.39%.
- Validation attack precision: 94.77%.
- Validation attack recall: 97.11%.
- Validation false-positive rate: 11.42%.
- Validation attack F1: 95.93%.
- Validation ROC AUC: 0.9872.
- Validation average precision: 0.9928.

Training and validation performance are similar, with no large gap.

Compared with logistic regression at the default threshold, the tree
produces 650 fewer false alarms but misses 403 additional attacks.
Its slightly higher summary scores do not resolve this tradeoff.

The official test set remains unused for model evaluation.